# XLEAP Detector

This notebook detects XLEAP using undulator taper, and quantifies the taper in MeV/fs, and reports which undulators are lasing.

## Theory (see Zhirong's book, Ch. 2.4)

In an undulator with magnetic field strength $B_0$ and wavenumber $k_u$, the electrons encounter a magnetic field in $\hat y$

$$
\vec{B}_u = B_0 \sin(k_u z) \hat y
$$

that produces a force in $\hat x$:

$$\begin{align*}
\dot {\vec p} = \partial_t (\gamma mc \vec \beta)&= -e (\cancel{\vec E} + \vec v \times \vec B) \\
&= -e(\beta c \hat z) \times B_0 \sin(k_u z) \hat y \\
&= +e\beta c B_0 \sin(k_u z) \hat x \\
\end{align*}
$$
Neglecting energy losses from steering radiation, we take $\dot \gamma = 0$ and integrate (taking $\beta_x = 0$ at $t=0$ from before the undulator) to find $\beta_x$:
$$
\begin{align*}
\dot \beta_x &= \frac{e \beta c B_0}{\gamma m c} \sin(k_u z) = \frac{e \beta B_0}{\gamma m} \sin(k_u \beta c t) \tag{$\beta_z \approx \beta \lesssim 1$} \\
\beta_x &= \left. \frac{e \beta B_0}{\gamma m k_u \beta c} \cos(k_u \beta c t)\right \vert_0^t \\
&= \frac{e B_0}{\gamma k_u m c} \cos(k_u \beta c t) \\
&= \frac{1}{\gamma}\underbrace{\frac{e B_0}{k_u m c}}_K \cos(k_u z) \coloneqq \boxed{\frac{K}{\gamma} \cos(k_u z)}

\end{align*}
$$

Again neglecting energy losses from steering radiation, we say that energy $\gamma mc^2$ and therefore total velocity $\beta$ is conserved from before the undulator, but once inside the undulator, the velocity is split between $x$ and $z$ components:

$$
\begin{align*}
{\vec \beta}^2 &= \beta_z^2 + \beta_x^2 \\
\beta_z^2 &= \vec \beta^2 - \beta_x^2 = \left(1 - \frac {1}{\gamma^2} \right) - \beta_x^2 \\
\beta_z &= \sqrt{1-\frac{1}{\gamma^2}-\beta_x^2} \\ 
&\approx 1- \frac{1}{2\gamma^2} - \frac{K^2}{2\gamma^2}\cos^2(k_u z) \tag{$\sqrt{1-\varepsilon}\approx 1-\varepsilon/2$} \\
\langle \beta_z \rangle &= 1- \frac{1}{2\gamma^2} - \frac{K^2}{2\gamma^2}\left\langle\cos^2({k_u z})\right\rangle = 1- \frac{1}{2\gamma^2} - \frac{K^2}{4\gamma^2} \\
&= 1- \left(\frac{1+K^2/2}{2\gamma^2}\right)
\end{align*}
$$
So the light outruns the electrons by $\Delta \beta = (1+K^2/2)/2\gamma^2$. To achieve resonance, we want the radiation to slip ahead of the electrons by one radiation wavelength $\lambda_r$ per undulator period $\lambda_u$. Light covers an undulator period in time $T = \lambda_u/c$, during which time the electrons fall behind by a distance $\Delta \beta c T = \lambda_u(1+K^2/2)/2\gamma^2$, called the **slippage length**. Therefore we set the radiation wavelength equal to the slippage length so that the radiation and undulator fields remain in phase:
$$
\lambda_r = \frac{\lambda_u}{2\gamma^2}\left(1+\frac{K^2}{2}\right)
$$

Meanwhile, space charge forces create an energy spread within the current spike by pushing head electrons forward and tail electrons backward. The radiation slips from lower-energy to higher-energy slices of electrons: $\gamma \to \gamma + \Delta \gamma$. Due to the taper, $K$ also increases downstream: $K \to K + \Delta K$. To maintain resonance, we now need
$$
\lambda_r = \frac{\lambda_u}{2(\gamma + \Delta \gamma)^2}\left(1+\frac{(K + \Delta K)^2}{2}\right)
$$

Therefore, given a $\Delta K$, we can find the corresponding $\Delta \gamma$ needed to maintain resonance:
$$
\begin{align*}
\lambda_r = \frac{\lambda_u}{2\gamma^2}\left(1+\frac{K^2}{2}\right) &= \frac{\lambda_u}{2(\gamma+\Delta \gamma)^2}\left(1+\frac{(K+\Delta K)^2}{2}\right) \\
\implies \Delta \gamma &= \gamma\left(\sqrt{1+\frac{(K+\Delta K/2)\Delta K}{1+K^2/2}}-1\right) \\
&\approx \gamma\left(\frac{K\Delta K + \Delta K^2/2}{K^2+2}\right) \sim \gamma\left(\frac{\Delta K}{K} + \mathcal{O}\left(\left[\frac{\Delta K}{K}\right]^2\right)\right)
\end{align*}
$$

## Implementation

We do a "coarse sweep" between `start` and `end` times, taking snapshots at intervals of `coarse_delta`.
We are interested in three things:
1. Whether XLEAP is happening, indicated by the presence of at least one lasing tapered group of undulators;
2. Which undulators are lasing, as a boolean array (or number of undulators, if we sum);
3. The undulator taper in MeV/fs, calculated by
    
    a) taking the median of the sorted list of $\Delta K$ within the lasing group, 
    
    b) converting to $\Delta \gamma$, and
    
    c) dividing by one undulator length in units of time, $L_u/c$


In [1]:
import meme
import os, re
import numpy as np
from datetime import timedelta, datetime as dt

start = dt(2026, 6, 1, 0, 0)
end = dt.now()
coarse_delta = timedelta(hours=4)
snapshot_delta = timedelta(seconds=3)

## Data collection

We get $\lambda_u$ as a per-undulator PV field ending in `:LAMBDA_U`. We take the length of an undulator $L_u$ as 3.4 m, ignoring the spaces between undulators since negligible slippage occurs without the magnetic field bleeding $\beta_z$ into $\beta_x$.

We get $\gamma$ from the beam dump PV `BEND:DMPS:400:BACT`, which reports the strength of the beam dump magnet not in teslas, but in **GeV/c**, the momentum of beam energy being deflected ($\gamma mc^2 / c = \gamma mc$).

Finally, we take a snapshot of the series of undulator $K$ values. We take $K$ as the first $K$ in the most upstream "hockey stick" region -- that is, a group of at least $n=4$ undulators whose $K$'s are increasing by at least $4\rho$ per undulator. These are discovered by binary erosion, explained in the code below. 



In [5]:
def _fetch_archiver_snapshot(pv_names, from_time, to_time):
    if isinstance(pv_names, str):
        pv_names = [pv_names]
    payload = meme.archive.get(pv_names, from_time=from_time, to_time=to_time)
    if len(pv_names) == 1:
        pv = pv_names[0]
        return {pv: (dt.fromtimestamp(payload['secondsPastEpoch'][-1]), payload['values'][-1])}
    else:
        # return payload[0]
        return {d['pvName']: (dt.fromtimestamp(d['value']['value']['secondsPastEpoch'][-1]), d['value']['value']['values'][-1]) for d in payload}

In [7]:
_fetch_archiver_snapshot(['BEND:DMPS:400:BACT'], from_time=end, to_time=end + snapshot_delta)

{'BEND:DMPS:400:BACT': (datetime.datetime(2026, 6, 13, 18, 31, 41),
  np.float64(4.999996396518345))}

### $\lambda_u$

In [8]:

# Discover all undulators
meme.names.list_pvs('USEG:UNDS:%:KAct')

2026-06-13T18:32:15.743768024 WARN pvxs.client.io Server 134.79.151.36:42467 no supported auth.  try to force 'anonymous'


['USEG:UNDS:2150:KAct',
 'USEG:UNDS:2250:KAct',
 'USEG:UNDS:2350:KAct',
 'USEG:UNDS:2450:KAct',
 'USEG:UNDS:2550:KAct',
 'USEG:UNDS:2650:KAct',
 'USEG:UNDS:2750:KAct',
 'USEG:UNDS:2850:KAct',
 'USEG:UNDS:2950:KAct',
 'USEG:UNDS:3050:KAct',
 'USEG:UNDS:3150:KAct',
 'USEG:UNDS:3250:KAct',
 'USEG:UNDS:3350:KAct',
 'USEG:UNDS:3450:KAct',
 'USEG:UNDS:3650:KAct',
 'USEG:UNDS:3750:KAct',
 'USEG:UNDS:3850:KAct',
 'USEG:UNDS:3950:KAct',
 'USEG:UNDS:4050:KAct',
 'USEG:UNDS:4150:KAct',
 'USEG:UNDS:4250:KAct',
 'USEG:UNDS:4350:KAct',
 'USEG:UNDS:4450:KAct',
 'USEG:UNDS:4550:KAct',
 'USEG:UNDS:4650:KAct',
 'USEG:UNDS:4750:KAct']

In [9]:
# Find the lambda_U field in the undulator PVs
meme.names.list_pvs('USEG:UNDS:4750:%')

2026-06-13T18:32:18.977315573 ERR pvxs.client.dup Duplicate PV name ds from 134.79.151.36:42467 and 134.79.151.21:34077
2026-06-13T18:32:18.977515810 WARN pvxs.client.io Server 134.79.151.36:42467 no supported auth.  try to force 'anonymous'


['USEG:UNDS:4750:ACKALL',
 'USEG:UNDS:4750:AD:STATSUMY',
 'USEG:UNDS:4750:AD:STATSUMYFP',
 'USEG:UNDS:4750:AG2KSpl:Check',
 'USEG:UNDS:4750:AG2KSpl:CheckStat',
 'USEG:UNDS:4750:AG2KSpl:DeltaKTemp',
 'USEG:UNDS:4750:AG2KSpl:InputTempCor',
 'USEG:UNDS:4750:AG2KSpl:Limits',
 'USEG:UNDS:4750:AG2KSpl:MaxGAP',
 'USEG:UNDS:4750:AG2KSpl:MaxK',
 'USEG:UNDS:4750:AG2KSpl:MinGAP',
 'USEG:UNDS:4750:AG2KSpl:MinK',
 'USEG:UNDS:4750:AG2KSpl:OutputTempCor',
 'USEG:UNDS:4750:AG2KSpl:TempIn',
 'USEG:UNDS:4750:AG2KSpl:TempOut',
 'USEG:UNDS:4750:AG2KSpl:Trans1',
 'USEG:UNDS:4750:AG2KSpl:Trans1Dir',
 'USEG:UNDS:4750:AG2KSpl:Trans1Name',
 'USEG:UNDS:4750:AUTOBRAKE',
 'USEG:UNDS:4750:Abort',
 'USEG:UNDS:4750:AbortState',
 'USEG:UNDS:4750:ActiveMode',
 'USEG:UNDS:4750:AlignmentOffset',
 'USEG:UNDS:4750:AlignmentSlope',
 'USEG:UNDS:4750:AlignmentTaper',
 'USEG:UNDS:4750:AllowUndCenter',
 'USEG:UNDS:4750:AuotofocusEnabledDbg',
 'USEG:UNDS:4750:AutoCommit',
 'USEG:UNDS:4750:CM:STATSUMY',
 'USEG:UNDS:4750:CM:STATS

In [10]:
# Discover all undulator periods
lambda_u_pvs = meme.names.list_pvs('USEG:UNDS:%:lambda_U')
lambda_u_dict = {}
for pv in lambda_u_pvs:
    payload = meme.archive.get(pv)
    lambda_u_dict[pv] = payload['values']
    print(pv, payload)


USEG:UNDS:2150:lambda_U {'secondsPastEpoch': array([], dtype=float64), 'values': array([], dtype=float64), 'nanoseconds': array([], dtype=float64), 'severity': array([], dtype=float64), 'status': array([], dtype=float64)}
USEG:UNDS:2250:lambda_U {'secondsPastEpoch': array([], dtype=float64), 'values': array([], dtype=float64), 'nanoseconds': array([], dtype=float64), 'severity': array([], dtype=float64), 'status': array([], dtype=float64)}
USEG:UNDS:2350:lambda_U {'secondsPastEpoch': array([], dtype=float64), 'values': array([], dtype=float64), 'nanoseconds': array([], dtype=float64), 'severity': array([], dtype=float64), 'status': array([], dtype=float64)}
USEG:UNDS:2450:lambda_U {'secondsPastEpoch': array([], dtype=float64), 'values': array([], dtype=float64), 'nanoseconds': array([], dtype=float64), 'severity': array([], dtype=float64), 'status': array([], dtype=float64)}
USEG:UNDS:2550:lambda_U {'secondsPastEpoch': array([], dtype=float64), 'values': array([], dtype=float64), 'nano

In [11]:
lambda_u_dict

{'USEG:UNDS:2150:lambda_U': array([], dtype=float64),
 'USEG:UNDS:2250:lambda_U': array([], dtype=float64),
 'USEG:UNDS:2350:lambda_U': array([], dtype=float64),
 'USEG:UNDS:2450:lambda_U': array([], dtype=float64),
 'USEG:UNDS:2550:lambda_U': array([], dtype=float64),
 'USEG:UNDS:2650:lambda_U': array([0.056]),
 'USEG:UNDS:2750:lambda_U': array([0.056]),
 'USEG:UNDS:2850:lambda_U': array([0.056]),
 'USEG:UNDS:2950:lambda_U': array([0.056]),
 'USEG:UNDS:3050:lambda_U': array([0.056]),
 'USEG:UNDS:3150:lambda_U': array([0.056]),
 'USEG:UNDS:3250:lambda_U': array([0.056]),
 'USEG:UNDS:3350:lambda_U': array([0.056]),
 'USEG:UNDS:3450:lambda_U': array([0.056]),
 'USEG:UNDS:3650:lambda_U': array([0.056]),
 'USEG:UNDS:3750:lambda_U': array([0.056]),
 'USEG:UNDS:3850:lambda_U': array([0.056]),
 'USEG:UNDS:3950:lambda_U': array([0.056]),
 'USEG:UNDS:4050:lambda_U': array([0.056]),
 'USEG:UNDS:4150:lambda_U': array([0.056]),
 'USEG:UNDS:4250:lambda_U': array([0.056]),
 'USEG:UNDS:4350:lambda_U'

For some reason $\lambda_u$ is not archived for some undulators, so we use a cheeky `caget` just to be safe:

In [13]:
for pv in lambda_u_pvs:
    os.system(f'caget {pv}')

USEG:UNDS:2150:lambda_U        0.056
USEG:UNDS:2250:lambda_U        0.056
USEG:UNDS:2350:lambda_U        0.056
USEG:UNDS:2450:lambda_U        0.056
USEG:UNDS:2550:lambda_U        0.056
USEG:UNDS:2650:lambda_U        0.056
USEG:UNDS:2750:lambda_U        0.056
USEG:UNDS:2850:lambda_U        0.056
USEG:UNDS:2950:lambda_U        0.056
USEG:UNDS:3050:lambda_U        0.056
USEG:UNDS:3150:lambda_U        0.056
USEG:UNDS:3250:lambda_U        0.056
USEG:UNDS:3350:lambda_U        0.056
USEG:UNDS:3450:lambda_U        0.056
USEG:UNDS:3650:lambda_U        0.056
USEG:UNDS:3750:lambda_U        0.056
USEG:UNDS:3850:lambda_U        0.056
USEG:UNDS:3950:lambda_U        0.056
USEG:UNDS:4050:lambda_U        0.056
USEG:UNDS:4150:lambda_U        0.056
USEG:UNDS:4250:lambda_U        0.056
USEG:UNDS:4350:lambda_U        0.039
USEG:UNDS:4450:lambda_U        0.039
USEG:UNDS:4550:lambda_U        0.039
USEG:UNDS:4650:lambda_U        0.039
USEG:UNDS:4750:lambda_U        0.039


In [14]:
# Use caget to populate the lambda_u_dict with the current values
for pv in lambda_u_pvs:
    value = os.popen(f'caget {pv}').read().strip()
    value = float(re.search(r'[0-9.-]+$', value).group()) # Find a decimal number at the end of the string
    lambda_u_dict[pv] = value

In [15]:
# Fully populated thanks to caget
lambda_u_dict

{'USEG:UNDS:2150:lambda_U': 0.056,
 'USEG:UNDS:2250:lambda_U': 0.056,
 'USEG:UNDS:2350:lambda_U': 0.056,
 'USEG:UNDS:2450:lambda_U': 0.056,
 'USEG:UNDS:2550:lambda_U': 0.056,
 'USEG:UNDS:2650:lambda_U': 0.056,
 'USEG:UNDS:2750:lambda_U': 0.056,
 'USEG:UNDS:2850:lambda_U': 0.056,
 'USEG:UNDS:2950:lambda_U': 0.056,
 'USEG:UNDS:3050:lambda_U': 0.056,
 'USEG:UNDS:3150:lambda_U': 0.056,
 'USEG:UNDS:3250:lambda_U': 0.056,
 'USEG:UNDS:3350:lambda_U': 0.056,
 'USEG:UNDS:3450:lambda_U': 0.056,
 'USEG:UNDS:3650:lambda_U': 0.056,
 'USEG:UNDS:3750:lambda_U': 0.056,
 'USEG:UNDS:3850:lambda_U': 0.056,
 'USEG:UNDS:3950:lambda_U': 0.056,
 'USEG:UNDS:4050:lambda_U': 0.056,
 'USEG:UNDS:4150:lambda_U': 0.056,
 'USEG:UNDS:4250:lambda_U': 0.056,
 'USEG:UNDS:4350:lambda_U': 0.039,
 'USEG:UNDS:4450:lambda_U': 0.039,
 'USEG:UNDS:4550:lambda_U': 0.039,
 'USEG:UNDS:4650:lambda_U': 0.039,
 'USEG:UNDS:4750:lambda_U': 0.039}

### $\gamma$

In [16]:
# Runtime: around 10s for 10 days of data at 4h intervals, with 3s snapshots

gamma_snapshots = []
t = start
while t < end:
    snapshot = _fetch_archiver_snapshot('BEND:DMPS:400:BACT', from_time=t, to_time=t + snapshot_delta)
    gamma_snapshots.append(snapshot)
    t += coarse_delta

In [17]:
gamma_snapshots

[{'BEND:DMPS:400:BACT': (datetime.datetime(2026, 6, 1, 0, 0, 2),
   np.float64(10.000011194073167))},
 {'BEND:DMPS:400:BACT': (datetime.datetime(2026, 6, 1, 4, 0, 1),
   np.float64(9.999996080362873))},
 {'BEND:DMPS:400:BACT': (datetime.datetime(2026, 6, 1, 8, 0, 2),
   np.float64(9.999997371432526))},
 {'BEND:DMPS:400:BACT': (datetime.datetime(2026, 6, 1, 12, 0, 2),
   np.float64(10.000004732854245))},
 {'BEND:DMPS:400:BACT': (datetime.datetime(2026, 6, 1, 16, 0, 2),
   np.float64(9.999996898116873))},
 {'BEND:DMPS:400:BACT': (datetime.datetime(2026, 6, 1, 20, 0, 2),
   np.float64(10.000006359602438))},
 {'BEND:DMPS:400:BACT': (datetime.datetime(2026, 6, 2, 0, 0, 2),
   np.float64(10.000002237982937))},
 {'BEND:DMPS:400:BACT': (datetime.datetime(2026, 6, 2, 4, 0, 1),
   np.float64(10.000000932995325))},
 {'BEND:DMPS:400:BACT': (datetime.datetime(2026, 6, 2, 8, 0, 2),
   np.float64(9.999993393100754))},
 {'BEND:DMPS:400:BACT': (datetime.datetime(2026, 6, 2, 12, 0, 2),
   np.float64(9.9

### $K$

In [18]:
# This takes about a minute to pull data for 10 days. We will maintain a cache in a CSV file so we don't have to do this every time.



k_archive = {}
und_nums = [re.search(r'USEG:UNDS:(\d+):lambda_U', pv).group(1) for pv in lambda_u_pvs]
kact_pvs = [f'USEG:UNDS:{num}:KAct' for num in und_nums]
k_snapshots = []
t = start
while t < end:
    snapshot = _fetch_archiver_snapshot(kact_pvs, from_time=t, to_time=t + snapshot_delta)
    k_snapshots.append(snapshot)
    t += coarse_delta

# Update the cache CSV file with the new snapshots
with open('k_snapshots.csv', 'w') as f:
    f.write('timestamp,value\n')
    for snapshot in gamma_snapshots:
        for pv, (timestamp, value) in snapshot.items():
            f.write(f'{timestamp},{value}\n')

In [19]:
k_snapshots

[{'USEG:UNDS:2150:KAct': (datetime.datetime(2026, 6, 1, 0, 0, 2),
   np.float64(6.725062129133953)),
  'USEG:UNDS:2250:KAct': (datetime.datetime(2026, 6, 1, 0, 0),
   np.float64(6.72696357135488)),
  'USEG:UNDS:2350:KAct': (datetime.datetime(2026, 5, 31, 23, 59, 56),
   np.float64(6.726536693442391)),
  'USEG:UNDS:2450:KAct': (datetime.datetime(2026, 6, 1, 0, 0, 2),
   np.float64(6.724772491594098)),
  'USEG:UNDS:2550:KAct': (datetime.datetime(2026, 5, 31, 23, 59, 36),
   np.float64(6.724057586925176)),
  'USEG:UNDS:2650:KAct': (datetime.datetime(2026, 6, 1, 0, 0, 2),
   np.float64(6.7249611294244485)),
  'USEG:UNDS:2750:KAct': (datetime.datetime(2026, 6, 1, 0, 0, 2),
   np.float64(6.724410226463157)),
  'USEG:UNDS:2850:KAct': (datetime.datetime(2026, 5, 31, 23, 59, 59),
   np.float64(6.724680969380071)),
  'USEG:UNDS:2950:KAct': (datetime.datetime(2026, 6, 1, 0, 0),
   np.float64(6.7243741335124305)),
  'USEG:UNDS:3050:KAct': (datetime.datetime(2026, 5, 31, 23, 59, 51),
   np.float64(

## XLEAP detection
We say that XLEAP is happening when:
>There exists a group of at least 4 undulators whose $K$'s are increasing by at least $4\rho$ per undulator ($\rho \sim 0.3$).

To find such a group, we make a boolean array from the K array indicating where $K_{i}\geq K_{i-1}+4\rho$. Then we use "binary erosion" and "binary expansion" to filter for runs of at least $n=4$ consecutive increases of $\Delta K \geq 4\rho$.

In [20]:
# Build a display DataFrame from the K snapshots using nominal request times.
import pandas as pd

und_nums = [re.search(r'USEG:UNDS:(\d+):KAct', pv).group(1) for pv in kact_pvs]
nominal_times = [start + i * coarse_delta for i in range(len(k_snapshots))]

kvals_df = pd.DataFrame(
    data=[
        [snapshot[pv][1] if pv in snapshot else np.nan for pv in kact_pvs]
        for snapshot in k_snapshots
    ],
    index=nominal_times,
    columns=und_nums,
)
kvals_df.index.name = 'NOMINAL time'
kvals_df.columns.name = 'Undulator'
kvals_df

Undulator,2150,2250,2350,2450,2550,2650,2750,2850,2950,3050,...,3850,3950,4050,4150,4250,4350,4450,4550,4650,4750
NOMINAL time,,,,,,,,,,,,,,,,,,,,,
2026-06-01 00:00:00,6.725062,6.726964,6.726537,6.724772,6.724058,6.724961,6.724410,6.724681,6.724374,6.724943,...,6.718377,6.715974,6.711830,6.704956,1.372594,1.800388,1.321534,1.706796,1.753548,1.608738
2026-06-01 04:00:00,6.725048,6.727050,6.726618,6.724772,6.724007,6.725005,6.724353,6.724656,6.724377,6.724938,...,6.718459,6.716062,6.711892,6.705037,1.372595,1.800382,1.321519,1.706790,1.753538,1.608735
2026-06-01 08:00:00,6.725297,6.727509,6.727144,6.725134,6.724268,6.725200,6.724302,6.724814,6.724317,6.725221,...,6.718902,6.716551,6.712324,6.705563,0.000009,1.800506,1.321593,1.707004,1.753619,1.608807
2026-06-01 12:00:00,6.725227,6.727591,6.727217,6.725078,6.724149,6.725135,6.724231,6.724723,6.724106,6.725038,...,6.719045,6.716647,6.712492,6.705936,0.000009,1.800545,1.321635,1.707044,1.753689,1.608852
2026-06-01 16:00:00,6.725010,6.727479,6.727086,6.724912,6.723983,6.725029,6.724034,6.724583,6.723920,6.724834,...,6.718877,6.716580,6.712411,6.705922,5.273854,1.800589,1.321632,1.707032,1.753689,1.608853
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-13 00:00:00,4.236782,2.172589,5.302483,2.522315,4.087217,4.087118,4.087115,4.105844,4.124674,4.143247,...,2.798312,2.802503,2.805829,2.809314,2.812814,3.501500,3.505684,1.286586,1.758174,2.227364
2026-06-13 04:00:00,6.150169,6.322660,6.295147,6.318123,6.427117,6.426067,6.425407,6.424751,6.424281,6.423578,...,4.293625,4.126488,4.197627,4.323471,4.292008,5.280837,4.719855,4.718880,5.247307,4.714026
2026-06-13 08:00:00,5.830725,5.829638,5.828500,5.827533,5.826258,5.825119,5.824007,5.822844,5.821774,5.820751,...,5.754611,5.736480,5.715066,5.691484,5.665632,1.226410,1.082804,1.203852,1.196448,1.048642


In [21]:
# Binary erosion tests for intuition
from scipy.ndimage import binary_erosion, binary_dilation
def mask_n(arr, n=4):
    structure = [1] * n
    return binary_dilation(
        binary_erosion(arr, structure=structure),
        structure=structure,
    )

mask_n(np.array([0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1])).astype(int)

array([0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1])

In [23]:
def dk_to_dgamma(dK, K, gamma):
    return gamma * (np.sqrt(1 + ((K + dK / 2) * dK) / (1 + K**2 / 2)) - 1)

def k_to_dk(kvals_df: pd.DataFrame):
    # Calculate dK by subtracting K from a rolled version of itself (shifted by 1)
    dK_df = kvals_df - kvals_df.shift(1, axis=1)
    return dK_df

def detect_lasing_groups(kvals_df, rho=1e-3, rho_scale=4, n=4):
    '''
    1. Calculate dK by subtracting K from a rolled version of itself (shifted by 1)
    2. Mask dK >= rho * rho_scale
    3. Apply binary erosion and dilation to find contiguous groups of lasing undulators, with a minimum group size of n
    
    '''
    # 1. Calculate dK
    dk = k_to_dk(kvals_df)

    #2. Mask dK >= rho * rho_scale
    at_least_rho = dk >= rho * rho_scale

    # 3. Apply binary erosion and dilation row-by-row along the undulator axis.
    structure = [1] * n
    eroded_and_dilated = at_least_rho.apply(
        lambda row: binary_dilation(
            binary_erosion(row.to_numpy(dtype=bool), structure=structure),
            structure=structure,
        ),
        axis=1,
        result_type='expand',
    )
    eroded_and_dilated.index = kvals_df.index
    eroded_and_dilated.columns = kvals_df.columns
    return eroded_and_dilated

# Lasing group masks are built from dK arrays; add 1 to the left of each group to correct fencepost error when masking K array
dk_lasing_mask = detect_lasing_groups(kvals_df)
dk_group_left_edges = dk_lasing_mask & ~dk_lasing_mask.shift(1, axis=1, fill_value=False)
k_lasing_mask = dk_lasing_mask | dk_group_left_edges.shift(-1, axis=1, fill_value=False)

# Mask K values by lasing groups
lasing_groups = kvals_df[k_lasing_mask]
lasing_groups

Undulator,2150,2250,2350,2450,2550,2650,2750,2850,2950,3050,...,3850,3950,4050,4150,4250,4350,4450,4550,4650,4750
NOMINAL time,,,,,,,,,,,,,,,,,,,,,
2026-06-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-06-01 04:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-06-01 08:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-06-01 12:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-06-01 16:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-13 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,4.087115,4.105844,4.124674,4.143247,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-06-13 04:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-06-13 08:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


After coarse detection of lasing groups, search two hours to either side of each timestamp in which lasing groups appear. Use intervals of `fine_delta` (default: 15 minutes) to fill in the gaps.



In [24]:
fine_delta = globals().get('fine_delta', timedelta(minutes=15))
fine_window = timedelta(hours=2)

coarse_lasing_times = list(lasing_groups.index[lasing_groups.notna().any(axis=1)])

fine_times = set()
for coarse_time in coarse_lasing_times:
    t = coarse_time - fine_window
    while t <= coarse_time + fine_window:
        if start <= t <= end:
            fine_times.add(t)
        t += fine_delta
fine_times = sorted(fine_times)

fine_k_snapshots = [
    _fetch_archiver_snapshot(kact_pvs, from_time=t, to_time=t + snapshot_delta)
    for t in fine_times
]

fine_kvals_df = pd.DataFrame(
    data=[
        [snapshot[pv][1] if pv in snapshot else np.nan for pv in kact_pvs]
        for snapshot in fine_k_snapshots
    ],
    index=fine_times,
    columns=und_nums,
)
fine_kvals_df.index.name = 'NOMINAL time'
fine_kvals_df.columns.name = 'Undulator'

fine_snapshot_times_df = pd.DataFrame(
    data=[
        [snapshot[pv][0] if pv in snapshot else pd.NaT for pv in kact_pvs]
        for snapshot in fine_k_snapshots
    ],
    index=fine_times,
    columns=und_nums,
)
fine_snapshot_times_df.index.name = 'NOMINAL time'
fine_snapshot_times_df.columns.name = 'Undulator'

fine_dk_lasing_mask = detect_lasing_groups(fine_kvals_df)
fine_dk_group_left_edges = fine_dk_lasing_mask & ~fine_dk_lasing_mask.shift(1, axis=1, fill_value=False)
fine_k_lasing_mask = fine_dk_lasing_mask | fine_dk_group_left_edges.shift(-1, axis=1, fill_value=False)
fine_lasing_groups = fine_kvals_df[fine_k_lasing_mask]

fine_k_snapshots_df = pd.concat(
    {'K value': fine_kvals_df, 'Snapshot time': fine_snapshot_times_df},
    axis=1,
)
fine_k_snapshots_df.columns.names = ['Field', 'Undulator']

fine_lasing_groups

Undulator,2150,2250,2350,2450,2550,2650,2750,2850,2950,3050,...,3850,3950,4050,4150,4250,4350,4450,4550,4650,4750
NOMINAL time,,,,,,,,,,,,,,,,,,,,,
2026-06-09 22:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-06-09 22:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-06-09 22:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-06-09 22:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-06-09 23:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.316114,4.333758,4.351396,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-13 17:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.311412,5.335339,5.359246,...,NaN,3.712459,3.716944,3.721566,3.726185,4.061869,NaN,NaN,NaN,NaN
2026-06-13 17:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.231503,5.254424,5.277455,...,NaN,3.649225,3.653480,3.657968,3.662337,4.006416,NaN,NaN,NaN,NaN
2026-06-13 17:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.281128,5.304692,5.328169,...,NaN,3.696383,3.700838,3.705432,3.709979,4.047846,NaN,NaN,NaN,NaN


# Taper reporting
We take the median change in K of the first undulator group, convert to $\Delta \gamma$, and divide by one undulator length in units of time:
$$
T_u = \frac{L_u}{c} \approx \frac{3.4\text{ m}}{3\cdot 10^8 \text{ m/s}} \sim 1.13\cdot10^{-8}\text{ s} = 11.3\text{ ns}
$$

In [ ]:
L_u = 3.4 # Undulator length in meters
c = 299792458 # Speed of light in m/s
T_u = L_u / c # Undulator period in seconds

# isolate the first lasing group (= contiguous GROUP of undulators with dK > threshold) for each time
for time, undulator_values in fine_lasing_groups.iterrows():
    lasing_undulators = undulator_values.index[undulator_values.notna()]
    if len(lasing_undulators) == 0:
        continue
    first_group_start = lasing_undulators[0]
    first_group_end = first_group_start
    for und in lasing_undulators[1:]:
        if und == first_group_end + 1:
            first_group_end = und
        else:
            break
    print(f'Time: {time}, First lasing group: Undulators {first_group_start} to {first_group_end}')

AttributeError: 'tuple' object has no attribute 'notna'

# XLEAP Data Server
Using the above, we declare a function that takes in a start and end time and returns a list of data objects every quarter-hour between start and end times, including the precise start and end times. The data object contains:
```
{
    'xleap_on': bool            # False if no lasing undulator groups are detected; true otherwise
    'K_lasing': dict(str,float) # map from lasing und number to K (only lasing unds shown)
    'n_und':    int             # len(K_lasing)
    'taper':    float           # Taper of FIRST lasing group using median $\Delta K$ in MeV/fs
}
```
